In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.models.anomaly_methods import compare_methods, run_isolation_forest

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
%matplotlib inline

ModuleNotFoundError: No module named 'src'

In [ ]:
contacts = pd.read_parquet(PROCESSED_DIR / "contacts.parquet")

# A contact between A and B counts for both A and B
both = pd.concat([
    contacts.rename(columns={"user_a": "user", "user_b": "other"}),
    contacts.rename(columns={"user_b": "user", "user_a": "other"}),
]).drop_duplicates(subset=["timestamp", "user", "other"])
both["day"] = both["timestamp"] // 86_400

df = (
    both.groupby(["user", "day"])
    .agg(
        n_contact_records=("other", "size"),
        n_unique_contacts=("other", "nunique"),
        n_active_slots=("slot_id", "nunique"),
        median_rssi=("rssi", "median"),
    )
    .reset_index()
)

FEATURES = ["n_contact_records", "n_unique_contacts", "n_active_slots", "median_rssi"]
print(f"{len(df):,} user-days")
df.head()

In [ ]:
ranked = run_isolation_forest(df, FEATURES, contamination=0.02)
print(f"Anomalies: {ranked['is_anomaly'].sum()} of {len(ranked)}")
ranked.head(20)